In [ ]:
import json
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.enum.text import WD_ALIGN_PARAGRAPH

NOTEBOOK_PATH = "KBHightPrediction_Research.ipynb"
OUTPUT_DOC    = "KBHeightPrediction_Code_Document.docx"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    nb = json.load(f)

code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]

doc = Document()

# ── Title ──────────────────────────────────────────────────────────────────────
title = doc.add_heading("KB Height Prediction – Code Reference", level=0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(
    f"Notebook: {NOTEBOOK_PATH}\n"
    f"Total code cells: {len(code_cells)}"
)
doc.add_paragraph()

# ── Helper: add a shaded code block ───────────────────────────────────────────
def add_code_block(doc, code_text):
    para = doc.add_paragraph()
    para.paragraph_format.space_before = Pt(2)
    para.paragraph_format.space_after  = Pt(2)
    para.paragraph_format.left_indent  = Inches(0.2)

    # light-grey shading on the paragraph
    pPr = para._p.get_or_add_pPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"),   "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"),  "F2F2F2")
    pPr.append(shd)

    run = para.add_run(code_text.rstrip())
    run.font.name = "Courier New"
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor(0x1F, 0x1F, 0x7A)   # dark-blue for code

# ── Add each code cell ─────────────────────────────────────────────────────────
for idx, cell in enumerate(code_cells, start=1):
    heading = doc.add_heading(f"Cell {idx}", level=2)
    heading.runs[0].font.color.rgb = RGBColor(0x2E, 0x74, 0xB5)

    source = "".join(cell["source"])
    if source.strip():
        add_code_block(doc, source)
    else:
        doc.add_paragraph("(empty cell)")

    doc.add_paragraph()   # spacer between cells

doc.save(OUTPUT_DOC)
print(f"✅  Document saved → {OUTPUT_DOC}  ({len(code_cells)} code cells exported)")
